# Lecture 18 — Ensembles: Forests and Boosting

This lab follows the discovery in `blog.md`: diversity can make many imperfect learners stronger together.

▶️ Run in Colab: https://colab.research.google.com/github/manish7725/deeplearning/blob/main/Lecture%2018%20-%20Ensembles%3A%20Forests%20and%20Boosting/notebook.ipynb


## 1. Problem

We will classify a small house dataset and compare one decision tree with a randomized forest. Then we will inspect a simple additive boosting process.


## 2. Prediction

Before running anything: will a single tree be more sensitive to a changed training sample than an ensemble of randomized trees?


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.metrics import accuracy_score, mean_squared_error

rng = np.random.default_rng(7)
X = np.array([[2,700],[2,900],[3,1000],[3,1200],[4,1300],[4,1500],[2,850],[3,1100],[4,1400],[5,1700]])
y = np.array([0,0,1,1,1,1,0,1,1,1])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=7, stratify=y)

tree = DecisionTreeClassifier(max_depth=3, random_state=7).fit(X_train, y_train)
forest = RandomForestClassifier(n_estimators=50, max_depth=3, random_state=7).fit(X_train, y_train)
tree_acc = accuracy_score(y_test, tree.predict(X_test))
forest_acc = accuracy_score(y_test, forest.predict(X_test))
print('tree accuracy:', tree_acc)
print('forest accuracy:', forest_acc)


## 4. Mathematics

For regression, an ensemble average is $\hat y(x)=\frac{1}{B}\sum_b f_b(x)$.

Boosting updates $f_{m+1}(x)=f_m(x)+\eta g_{m+1}(x)$.


In [ ]:
predictions = np.array([10., 12., 11., 13., 14.])
average = predictions.mean()
assert np.isclose(average, 12.0)
print('ensemble average:', average)

y_true = np.array([10., 12., 14.])
first = np.array([11., 11., 11.])
residual = y_true - first
second = np.array([-1., 1., 2.])
eta = 0.5
updated = first + eta * second
assert np.allclose(residual, [-1., 1., 3.])
assert np.allclose(updated, [10.5, 11.5, 12.])
print('residuals:', residual)
print('updated prediction:', updated)


In [ ]:
# First implementation: bootstrap + majority vote from scratch.
def bootstrap_indices(n, rng):
    return rng.integers(0, n, size=n)

votes = []
for seed in range(9):
    local_rng = np.random.default_rng(seed)
    idx = bootstrap_indices(len(X_train), local_rng)
    stump = DecisionTreeClassifier(max_depth=2, random_state=seed)
    stump.fit(X_train[idx], y_train[idx])
    votes.append(stump.predict(X_test))

votes = np.vstack(votes)
forest_vote = (votes.mean(axis=0) >= 0.5).astype(int)
print('votes shape:', votes.shape)
print('scratch forest accuracy:', accuracy_score(y_test, forest_vote))


In [ ]:
import matplotlib.pyplot as plt

sizes = [5, 20, 50, 100, 200]
scores = []
for n_trees in sizes:
    model = RandomForestClassifier(n_estimators=n_trees, max_depth=3, random_state=7)
    model.fit(X_train, y_train)
    scores.append(accuracy_score(y_test, model.predict(X_test)))

plt.figure()
plt.plot(sizes, scores, marker='o')
plt.xlabel('number of trees')
plt.ylabel('test accuracy')
plt.title('Forest accuracy as ensemble size changes')
plt.show()


In [ ]:
# Change exactly one variable: n_estimators.
# Predict first: should the score improve forever?
n_estimators = 5  # YOUR CHANGE HERE
model = RandomForestClassifier(n_estimators=n_estimators, max_depth=3, random_state=7)
model.fit(X_train, y_train)
print('accuracy:', accuracy_score(y_test, model.predict(X_test)))


## 10. Observe

Compare the measured scores with your prediction. More trees can stabilize an ensemble, but improvements are not guaranteed forever.


## 11. Explain

Randomization gives the ensemble members different training views and therefore different errors. Aggregation can reduce variance when those errors are not perfectly correlated.


In [ ]:
# Challenge
# Level 4: change only max_depth and predict whether validation accuracy will rise or fall.
# Level 5: implement your own weighted ensemble where each tree contributes a different weight.

max_depth = 2  # YOUR CODE HERE
challenge = RandomForestClassifier(n_estimators=50, max_depth=max_depth, random_state=7)
challenge.fit(X_train, y_train)
print('challenge accuracy:', accuracy_score(y_test, challenge.predict(X_test)))


## 13. Reflection

Mastery checklist: explain why cloned models add little; explain bootstrap sampling; explain why feature subsampling helps; derive the averaging variance result; explain how boosting differs from bagging.
